# Machine Learning: Gender Wage Gap Decomposition
**Author:** Antara
**Goal:** Quantify how much of the gender wage gap is explained by occupation/industry segregation vs. unexplained factors, using regression decomposition and Random Forest feature importance.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import sys
sys.path.insert(0, "..")
from analysis.utils import load_employment

df = load_employment()
print(f"Shape: {df.shape}")
df.head()

Shape: (1487548, 14)


,SERIAL,PERNUM,SEX,AGE,RACE,STATEFIP,OCC,IND,INCWAGE,PERWT,SEX_LABEL,RACE_LABEL,STATE_NAME,INDUSTRY
0,3800,1,1,64,1,1,8990,570,28000,11.0,Male,White,Alabama,Construction
1,3800,2,2,52,1,1,5740,8191,28000,12.0,Female,White,Alabama,Healthcare
2,3800,4,2,26,1,1,9141,6180,35200,15.0,Female,White,Alabama,Finance & Insurance
3,3803,1,1,64,1,1,1010,9480,130000,199.0,Male,White,Alabama,Other Services
4,3803,3,1,34,1,1,7240,7190,29100,208.0,Male,White,Alabama,Management


## Step 1: Baseline Regression (No Occupation/Industry)

In [2]:
baseline_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'INCWAGE']].copy()
baseline_encoded = pd.get_dummies(baseline_df, columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL'], drop_first=True)

X_baseline = baseline_encoded.drop(columns=['INCWAGE'])
y_baseline = baseline_encoded['INCWAGE']

model_baseline = LinearRegression()
model_baseline.fit(X_baseline, y_baseline)

sex_col = [c for c in X_baseline.columns if 'SEX_LABEL' in c][0]
baseline_gender_coef = model_baseline.coef_[list(X_baseline.columns).index(sex_col)]

print(f"Baseline R\u00b2: {model_baseline.score(X_baseline, y_baseline):.4f}")
print(f"Gender coefficient ({sex_col}): ${baseline_gender_coef:,.2f}")
print(f"This means: holding age, race, and state constant, men earn ${baseline_gender_coef:,.2f} more than women on average.")

Baseline R²: 0.0757
Gender coefficient (SEX_LABEL_Male): $24,748.73
This means: holding age, race, and state constant, men earn $24,748.73 more than women on average.


## Step 2: Full Regression (Adding Occupation + Industry)

In [3]:
full_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC', 'IND', 'INCWAGE']].copy()
full_df['OCC_GROUP'] = (full_df['OCC'] // 100) * 100
full_df['IND_GROUP'] = (full_df['IND'] // 1000) * 1000

full_encoded = pd.get_dummies(
    full_df.drop(columns=['OCC', 'IND']),
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC_GROUP', 'IND_GROUP'],
    drop_first=True
)

X_full = full_encoded.drop(columns=['INCWAGE'])
y_full = full_encoded['INCWAGE']

model_full = LinearRegression()
model_full.fit(X_full, y_full)

sex_col_full = [c for c in X_full.columns if 'SEX_LABEL' in c][0]
full_gender_coef = model_full.coef_[list(X_full.columns).index(sex_col_full)]

print(f"Full model R\u00b2: {model_full.score(X_full, y_full):.4f}")
print(f"Gender coefficient ({sex_col_full}): ${full_gender_coef:,.2f}")
print(f"After controlling for occupation and industry, men earn ${full_gender_coef:,.2f} more than women on average.")

Full model R²: 0.2585
Gender coefficient (SEX_LABEL_Male): $22,426.09
After controlling for occupation and industry, men earn $22,426.09 more than women on average.


## Step 3: Wage Gap Decomposition Summary

In [4]:
explained_amount = baseline_gender_coef - full_gender_coef
explained_pct = (explained_amount / baseline_gender_coef) * 100
unexplained_pct = 100 - explained_pct

print("="*60)
print("GENDER WAGE GAP DECOMPOSITION")
print("="*60)
print(f"Raw gender gap (baseline):        ${baseline_gender_coef:,.2f}")
print(f"Adjusted gender gap (with occ/ind): ${full_gender_coef:,.2f}")
print(f"Amount explained by occupation/industry: ${explained_amount:,.2f} ({explained_pct:.1f}%)")
print(f"Unexplained gap (potential bias/other factors): {unexplained_pct:.1f}%")

GENDER WAGE GAP DECOMPOSITION
Raw gender gap (baseline):        $24,748.73
Adjusted gender gap (with occ/ind): $22,426.09
Amount explained by occupation/industry: $2,322.64 (9.4%)
Unexplained gap (potential bias/other factors): 90.6%


## Interpretation

Only 9.4% of the gender wage gap is explained by which occupation and industry someone works in. The remaining 90.6% persists even after accounting for job type, race, age and state. In other words, women in the same broad occupation and industry as men still earn substantially less. This isn't simply because women are concentrated in lower-paying jobs. that's a much stronger and more specific claim than 'women earn less' and it direclty supportsa story about unexplained disparity (potentially reflecting bias, negotiation gaps, promotion patterns, part-tme/full-time differences or other unmeasured factors)

The R² values are quite low (0.076 baseline, 0.259 full model), meaning the model only explains about 26% of the variation in wages overall — that's normal for wage data (lots of individual variation), but it means the absolute dollar coefficients are more reliable as average effects across the whole population than as precise individual predictions. It's fine to report the gender coefficient and decomposition percentage, but I'd avoid implying the model is highly accurate at predicting any one person's wage.

The "unexplained" 90.6% almost certainly includes other unmeasured factors beyond pure discrimination — things like hours worked (full-time vs. part-time), years of experience, education level, and seniority within an occupation aren't in your IPUMS table, so they get folded into the "unexplained" bucket too. It's worth phrasing this carefully in your write-up, e.g. "after controlling for the variables available in this dataset" rather than claiming it's pure bias.

-----------------------------------
# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Use the same full feature set (age, race, state, sex, occupation, industry)
rf_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC', 'IND', 'INCWAGE']].copy()
rf_df['OCC_GROUP'] = (rf_df['OCC'] // 100) * 100
rf_df['IND_GROUP'] = (rf_df['IND'] // 1000) * 1000

rf_encoded = pd.get_dummies(
    rf_df.drop(columns=['OCC', 'IND']),
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC_GROUP', 'IND_GROUP'],
    drop_first=True
)

X = rf_encoded.drop(columns=['INCWAGE'])
y = rf_encoded['INCWAGE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"Random Forest Train R²: {train_score:.4f}")
print(f"Random Forest Test R²: {test_score:.4f}")

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Use a 200k row sample for faster iteration
rf_df_sample = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC', 'IND', 'INCWAGE']].sample(n=200000, random_state=42).copy()
rf_df_sample['OCC_GROUP'] = (rf_df_sample['OCC'] // 100) * 100
rf_df_sample['IND_GROUP'] = (rf_df_sample['IND'] // 1000) * 1000

rf_encoded = pd.get_dummies(
    rf_df_sample.drop(columns=['OCC', 'IND']),
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC_GROUP', 'IND_GROUP'],
    drop_first=True
)

X = rf_encoded.drop(columns=['INCWAGE'])
y = rf_encoded['INCWAGE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)

print(f"Random Forest Train R²: {train_score:.4f}")
print(f"Random Forest Test R²: {test_score:.4f}")

Random Forest Train R²: 0.2399
Random Forest Test R²: 0.2101


In [7]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(importances.head(15).to_string(index=False))

# Check where SEX ranks
sex_rank = importances.reset_index(drop=True)
sex_row = sex_rank[sex_rank['feature'].str.contains('SEX_LABEL')]
print(f"\nSEX_LABEL_Male rank: #{sex_rank[sex_rank['feature'] == sex_row['feature'].values[0]].index[0] + 1} out of {len(sex_rank)} features")
print(sex_row.to_string(index=False))

Top 15 Most Important Features:
              feature  importance
                  AGE    0.304471
       OCC_GROUP_3000    0.125489
       SEX_LABEL_Male    0.110850
       OCC_GROUP_2100    0.071485
       OCC_GROUP_1000    0.045916
        OCC_GROUP_400    0.044008
        OCC_GROUP_100    0.042257
        OCC_GROUP_800    0.029284
       IND_GROUP_6000    0.018991
       OCC_GROUP_4800    0.017461
       OCC_GROUP_9600    0.013872
       IND_GROUP_8000    0.011484
STATE_NAME_California    0.011146
       OCC_GROUP_3200    0.010622
       IND_GROUP_5000    0.009898

SEX_LABEL_Male rank: #3 out of 167 features
       feature  importance
SEX_LABEL_Male     0.11085


In [8]:
import plotly.express as px

top15 = importances.head(15).copy()
top15['is_gender'] = top15['feature'].str.contains('SEX_LABEL')

fig_rf = px.bar(
    top15,
    x='importance',
    y='feature',
    orientation='h',
    color='is_gender',
    color_discrete_map={True: '#d7191c', False: '#2c7bb6'},
    title='Top 15 Feature Importances: Random Forest Wage Prediction',
    labels={'importance': 'Feature Importance', 'feature': '', 'is_gender': 'Gender Feature'}
)
fig_rf.update_layout(yaxis={'categoryorder': 'total ascending'}, template='plotly_white')
fig_rf.show()